# SCRES-IA David Work Lab

<div style="padding:14px 18px; border-left:6px solid #1f77b4; background:#f3f8ff; border-radius:8px; margin:12px 0;">
<b>Purpose:</b> a portable Colab/Kaggle/local laboratory where David can audit Track A, run Track B, and edit PPO/DMLPA/SAC-style models without touching the core repository first.
</div>

<div style="padding:14px 18px; border-left:6px solid #2ca02c; background:#f4fff4; border-radius:8px; margin:12px 0;">
<b>Execution order:</b> Track A first, Track B second. Track A is the Garrido-original boundary lane; Track B is the current positive downstream-control lane.
</div>

Open from GitHub/Colab whenever possible so David always starts from the latest version:

- GitHub branch: `codex/garrido-replication-experiments`
- Notebook path: `notebooks/david_track_a_track_b_colab_kaggle_playground.ipynb`
- Colab URL pattern: `https://colab.research.google.com/github/Thom-320/scres-ia/blob/codex/garrido-replication-experiments/notebooks/david_track_a_track_b_colab_kaggle_playground.ipynb`

This Work Lab is intentionally verbose. It should be easy to edit, easy to audit, and hard to misuse silently.

What it gives David:

1. **Auto-updating repo setup** for Colab/Kaggle/local runs.
2. **Track A lab**: thesis-faithful 18D/19D DKANA contract, static headroom gate, and optional repair PPO.
3. **Track B lab**: canonical 8D downstream-control setup, PPO/DMLPA/SAC sandbox, official runner hooks, and reward/observation sweep hooks.
4. **Editable model area**: change PPO, DMLPA, add LSTM/RecurrentPPO or SAC experiments.
5. **Glossary**: all contracts, variables, rewards, observation versions, and metrics explained at the end.

The default settings are for **smoke/debug iteration**, not final paper confirmation. For claims, use dense CRN frontiers, full Garrido-style metrics, and 5+ seeds.

## Current scientific state, in plain language

### Track A: Garrido original decision family

Track A is the thesis-faithful or thesis-adjacent lane. It uses Garrido-style strategic variables:

- inventory / buffer decisions, usually around the thesis inventory periods `I0`, `I168`, `I336`, `I504`, `I672`, `I1344`;
- manufacturing capacity shifts `S1`, `S2`, `S3`;
- in richer Track A variants, per-operation buffers for `Op3`, `Op5`, and `Op9`.

What we learned so far:

- Track A is very useful scientifically because it tells us where RL **does not** add value.
- Dense static frontiers repeatedly captured the best achievable behavior with Garrido's buffer/shift variables.
- PPO could often learn the same plateau, especially after behavior cloning / warm-start, but it did not produce a robust publishable win over the best dense static frontier.
- Best-known Track A repair result was essentially a technical tie: dynamic around `0.155247` vs best static around `0.155254` in the per-op conflict campaign.
- Therefore Track A is currently a **boundary characterization**: neural learning value is frontier-dependent; if the action space does not reach the binding constraint, RL cannot invent headroom.

This notebook still includes Track A because David may want to audit whether Track A is truly closed, test a new architecture, or reproduce the decision contract.

### Track B: downstream dispatch control

Track B is the current positive lane. It is an operational extension: it exposes downstream dispatch controls that Garrido left fixed.

The key idea:

- The real bottleneck is downstream recovery / dispatch, especially around `Op10` and `Op12`.
- If the agent can control those dispatch multipliers, it can reduce backlog and recovery tail risk.
- The confirmed Track B result uses `track_b_v1`, observation `v7`, reward `control_v1`, risk level `adaptive_benchmark_v2`, horizon `h104`, 5 seeds, and dense static CRN comparison.

Best current paper-facing Track B baseline:

- action contract: `track_b_v1` with 8 continuous action dimensions;
- observation: `v7`;
- reward: `control_v1`;
- risk level: `adaptive_benchmark_v2`;
- horizon: `104` weekly decisions;
- primary metric: Garrido/Excel order-level ReT;
- secondary metrics: CVaR/tail, CTj/RPj/DPj, service loss, backlog, flow fill, Cobb-Douglas, and cost.

Do not call Track B thesis-faithful. It is a faithful extension of the DES bottleneck, not the original thesis decision surface.

## How to run this Work Lab

Recommended flow:

1. Run the **central configuration** cell.
2. Run **portable setup**. This cell auto-updates the local repo checkout by default.
3. Run the quick smoke tests.
4. Work through **Track A** first:
   - thesis-faithful DKANA smoke;
   - optional static headroom gate;
   - optional repair PPO only if the gate says there is headroom.
5. Work through **Track B** second:
   - inspect observation contracts;
   - optionally edit DMLPA/model code;
   - run the notebook-native PPO/DMLPA sandbox;
   - optionally call official Track B runners.
6. Save the manifest at the end.

### Auto-update behavior

The setup cell has three update controls:

- `AUTO_UPDATE_REPO=True`: fetch and fast-forward the selected GitHub branch when the notebook starts.
- `FORCE_RESET_REPO=False`: if set to `True`, overwrite local repo edits with `origin/<branch>`. Use this only in disposable Colab/Kaggle runtimes.
- `UPDATE_NOTEBOOK_COPY=False`: if set to `True`, download the latest notebook file into the repo checkout. This updates the file on disk; Colab does not safely hot-reload the already-open UI notebook, so reopen the notebook from GitHub/Colab to refresh visible cells.

This means the **code and local notebook copy can update automatically** in Colab/Kaggle. The already-open browser notebook cannot be magically replaced without reopening, which is a Colab/Kaggle UI limitation rather than a repo limitation.

In [ ]:
# ============================================================
# 0) Central configuration cell
# ============================================================
# Edit this cell first. Most experiments can be changed here.

from __future__ import annotations
from pathlib import Path
import os
import sys
import json
import platform
import subprocess
from datetime import datetime, timezone

# --- Repository setup ---
GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "codex/garrido-replication-experiments"

# Repo source options:
# - "auto": recommended. Uses the current folder when it is already a repo; otherwise clones from GitHub.
# - "git": clone/pull from GitHub into the platform-specific working directory.
# - "drive": copy a repo folder from Google Drive. Useful if David has a private working copy.
# - "upload_zip": unzip a repo archive already uploaded to the runtime.
# - "local": use the current working directory as the repo.
REPO_SOURCE = "auto"  # "auto" | "git" | "drive" | "upload_zip" | "local"
COLAB_REPO_PATH = "/content/scres-ia"
KAGGLE_REPO_PATH = "/kaggle/working/scres-ia"
LOCAL_REPO_PATH = COLAB_REPO_PATH  # Backward-compatible alias used by drive/upload paths.
DRIVE_REPO_PATH = "/content/drive/MyDrive/scres-ia"
UPLOAD_ZIP_PATH = "/content/scres-ia.zip"

# --- Auto-update controls ---
AUTO_UPDATE_REPO = True
FORCE_RESET_REPO = False
UPDATE_NOTEBOOK_COPY = False
NOTEBOOK_RELATIVE_PATH = "notebooks/david_track_a_track_b_colab_kaggle_playground.ipynb"
INSTALL_DEPS = True
SETUP_DRY_RUN = False  # True only for debugging path resolution; it skips clone/copy/install.

# --- Profile ladder ---
# EXPERIMENT_PROFILE controls which notebook blocks are on by default.
# RUN_PROFILE controls scale (timesteps/seeds/horizon).
EXPERIMENT_PROFILE = "track_b_smoke"  # custom | track_a_smoke | track_a_gate | track_a_dkana_export | track_b_smoke | track_b_debug | track_b_sweep | track_b_confirm_light
RUN_PROFILE = "smoke"  # smoke | debug | serious

PROFILE_LADDER = {
    "custom": "Use the manual toggles below.",
    "track_a_smoke": "Track A thesis-faithful 18D/19D smoke only.",
    "track_a_gate": "Track A static headroom gate; no blind PPO.",
    "track_a_dkana_export": "Export Track A DKANA trajectories and build windows.",
    "track_b_smoke": "Track B notebook-native PPO smoke; cheapest useful path.",
    "track_b_debug": "Track B notebook-native debug run with a little more signal.",
    "track_b_sweep": "Official Track B reward/observation sweep hook.",
    "track_b_confirm_light": "Light official Track B runner; still not paper-confirmatory.",
}

# Scale presets. Keep smoke tiny so Colab/Kaggle startup can be verified fast.
if RUN_PROFILE == "smoke":
    TRAIN_SEEDS = [1]
    TRAIN_TIMESTEPS = 512
    EVAL_EPISODES = 1
    MAX_STEPS = 8
    N_ENVS = 1
elif RUN_PROFILE == "debug":
    TRAIN_SEEDS = [1]
    TRAIN_TIMESTEPS = 5_000
    EVAL_EPISODES = 2
    MAX_STEPS = 26
    N_ENVS = 2
elif RUN_PROFILE == "serious":
    TRAIN_SEEDS = [1, 2]
    TRAIN_TIMESTEPS = 40_000
    EVAL_EPISODES = 4
    MAX_STEPS = 104
    N_ENVS = 4
else:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}")

# --- Work Lab execution toggles ---
RUN_TRACK_A_DKANA_SMOKE = True
RUN_TRACK_A_HEADROOM_GATE = False
RUN_TRACK_A_REPAIR = False
RUN_TRACK_A_DKANA_EXPORT = False
RUN_TRACK_A_DKANA_BUILD_WINDOWS = False
RUN_TRACK_A_DKANA_TRAIN_BC = False
RUN_TRACK_B_NOTEBOOK_SANDBOX = True
RUN_OFFICIAL_TRACK_B = False
RUN_TRACK_B_SWEEP = False
RUN_STATS_ANALYSIS = True
RUN_FIGURES = True
RUN_ARTIFACT_PACKAGER = True

if EXPERIMENT_PROFILE != "custom":
    RUN_TRACK_A_DKANA_SMOKE = EXPERIMENT_PROFILE in {"track_a_smoke", "track_a_gate", "track_a_dkana_export"}
    RUN_TRACK_A_HEADROOM_GATE = EXPERIMENT_PROFILE == "track_a_gate"
    RUN_TRACK_A_REPAIR = False
    RUN_TRACK_A_DKANA_EXPORT = EXPERIMENT_PROFILE == "track_a_dkana_export"
    RUN_TRACK_A_DKANA_BUILD_WINDOWS = EXPERIMENT_PROFILE == "track_a_dkana_export"
    RUN_TRACK_A_DKANA_TRAIN_BC = False
    RUN_TRACK_B_NOTEBOOK_SANDBOX = EXPERIMENT_PROFILE in {"track_b_smoke", "track_b_debug", "track_b_confirm_light"}
    RUN_OFFICIAL_TRACK_B = EXPERIMENT_PROFILE == "track_b_confirm_light"
    RUN_TRACK_B_SWEEP = EXPERIMENT_PROFILE == "track_b_sweep"

# --- Track A defaults ---
TRACK_A_DEFAULTS = {
    "lane": "boundary_characterization",
    "reward_mode": "ReT_excel_plus_cvar",
    "cvar_alpha": 0.1,
    "mode": "per_op",
    "families": "R13,R24",
    "phis": "1,2,4,8",
    "psis": "1.0",
    "fracs": "0,0.05,0.075,0.10,0.125,0.15,0.20,0.25",
    "shifts": "1,2,3",
    "max_steps": MAX_STEPS,
    "quick": True,
}
TRACK_A_FROZEN_REFERENCE = {
    "description": "Best-known Track A static/per-op plateau from repair experiments; dynamic did not beat it robustly.",
    "action_family": "per_op_buffer + categorical shift",
    "op3_frac": 0.0,
    "op5_frac": 0.10,
    "op9_frac": 0.0,
    "shift": "S2",
    "best_static_excel_ret_approx": 0.155254,
    "best_dynamic_excel_ret_approx": 0.155247,
    "interpretation": "technical tie; no publishable Track A dynamic headroom so far",
}

# Track A DKANA export defaults.
DKANA_OBSERVATION_MODE = "env_sdm_history_reward"  # decision_reward | env_reward | env_state_reward | env_sdm_history_reward
DKANA_OBSERVATION_VERSION = "v5"
DKANA_REWARD_MODE = "ReT_seq_v1"
DKANA_RISK_LEVEL = "increased"
DKANA_EPISODES = 2 if RUN_PROFILE == "smoke" else 10
DKANA_SEED_START = 0
DKANA_WINDOW_SIZE = 12
DKANA_RELATION_MODE = "temporal_delta"
DKANA_INCLUDE_PREV_REWARD = True
DKANA_STOCHASTIC_PT = True

# --- Track B defaults ---
TRACK_B_ENV_DEFAULTS = {
    "reward_mode": "control_v1",
    "risk_level": "adaptive_benchmark_v2",
    "observation_version": "v7",
    "action_contract": "track_b_v1",
    "step_size_hours": 168.0,
    "max_steps": MAX_STEPS,
}
TRACK_B_PPO_DEFAULTS = {
    "learning_rate": 1e-4,
    "n_steps": 256,
    "batch_size": 64,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
}
TRACK_B_DEFAULTS = {**TRACK_B_ENV_DEFAULTS, **TRACK_B_PPO_DEFAULTS}
TRACK_B_OBSERVATION_CANDIDATES = ["v7", "v8", "v9"]
TRACK_B_REWARD_CANDIDATES = ["control_v1", "ReT_excel_plus_cvar", "ReT_tail_v2", "ReT_garrido2024_train"]
TRACK_B_FROZEN_REFERENCE = {
    "description": "Canonical Track B confirmed result: PPO vs dense static CRN under adaptive_benchmark_v2.",
    "action_contract": "track_b_v1",
    "observation_version": "v7",
    "reward_mode": "control_v1",
    "risk_level": "adaptive_benchmark_v2",
    "horizon": "h104",
    "confirmed_train_seeds": 5,
    "confirmed_timesteps": 60_000,
    "order_ret_excel_ppo_approx": 0.00589,
    "order_ret_excel_best_static_approx": 0.00547,
    "delta_approx": 0.00043,
    "mechanism": "adaptive recovery / backlog control, not yet proven anticipation",
}

# --- Editable model defaults ---
# Available in this notebook-native sandbox:
# - ppo_mlp: baseline PPO MLP.
# - ppo_dmlpa_faithful: original David DMLPA architecture.
# - ppo_dmlpa_positional: DMLPA with sinusoidal positional encoding + LayerNorm.
# - sac_mlp: SAC baseline for continuous actions.
MODEL_KIND = "ppo_mlp"  # ppo_mlp | ppo_dmlpa_faithful | ppo_dmlpa_positional | sac_mlp
MODEL_KINDS_TO_RUN = ["ppo_mlp"] if MODEL_KIND == "ppo_mlp" else ["ppo_mlp", MODEL_KIND]
DMLPA_FACTOR = 1
DMLPA_FEATURES_DIM = 120
DMLPA_NHEAD = 12
DMLPA_NUM_LAYERS = 4
DMLPA_HIDDEN = 100

# --- Outputs ---
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = Path("outputs/david_playground") / f"{EXPERIMENT_PROFILE}_{RUN_PROFILE}_{RUN_TAG}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("CONFIG loaded")
print(json.dumps({
    "EXPERIMENT_PROFILE": EXPERIMENT_PROFILE,
    "RUN_PROFILE": RUN_PROFILE,
    "TRAIN_SEEDS": TRAIN_SEEDS,
    "TRAIN_TIMESTEPS": TRAIN_TIMESTEPS,
    "EVAL_EPISODES": EVAL_EPISODES,
    "MAX_STEPS": MAX_STEPS,
    "MODEL_KINDS_TO_RUN": MODEL_KINDS_TO_RUN,
    "AUTO_UPDATE_REPO": AUTO_UPDATE_REPO,
    "REPO_SOURCE": REPO_SOURCE,
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
}, indent=2))


In [ ]:
# ============================================================
# 1) Portable setup for Colab, Kaggle, Drive, zip upload, or local execution
# ============================================================
from pathlib import Path
import os
import sys
import shutil
import zipfile
import subprocess
import platform


def detect_platform() -> str:
    """Return one of: colab, kaggle, local.

    We intentionally check both environment variables and canonical folders because
    Colab/Kaggle sometimes expose one before the other during notebook startup.
    """
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists():
        # Kaggle images can have /content in unusual cases, so let Kaggle override below.
        platform_guess = "colab"
    else:
        platform_guess = "local"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").exists():
        return "kaggle"
    return platform_guess


def is_repo_dir(path: Path) -> bool:
    """A lightweight repo handle check used before cloning or copying."""
    return (path / "supply_chain").exists() and (path / "scripts").exists()


def run_cmd(cmd, *, cwd=None, check=False):
    print("$", " ".join(str(x) for x in cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-6000:], flush=True)
    if result.returncode != 0 and result.stderr:
        print(result.stderr[-6000:], flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result


PLATFORM = detect_platform()
print("Detected platform:", PLATFORM)

# Platform-specific default clone target. This is the handle that fixes the common
# /content vs /kaggle/working mismatch.
if PLATFORM == "kaggle":
    PLATFORM_REPO_PATH = Path(KAGGLE_REPO_PATH)
elif PLATFORM == "colab":
    PLATFORM_REPO_PATH = Path(COLAB_REPO_PATH)
else:
    PLATFORM_REPO_PATH = Path.cwd()

requested_source = REPO_SOURCE
if requested_source == "auto":
    # Local/dev notebook use should never clone into /content. If the current folder
    # already looks like the repo, use it. Otherwise Colab/Kaggle clone from GitHub.
    if is_repo_dir(Path.cwd()):
        RESOLVED_REPO_SOURCE = "local"
    else:
        RESOLVED_REPO_SOURCE = "git"
else:
    RESOLVED_REPO_SOURCE = requested_source

print("Requested REPO_SOURCE:", requested_source)
print("Resolved REPO_SOURCE:", RESOLVED_REPO_SOURCE)
print("Platform repo target:", PLATFORM_REPO_PATH)

if RESOLVED_REPO_SOURCE == "local":
    ROOT = Path.cwd().resolve()
    if not is_repo_dir(ROOT):
        raise FileNotFoundError(
            f"REPO_SOURCE='local' but current directory is not a scres-ia repo: {ROOT}. "
            "Open the notebook from the repo root or set REPO_SOURCE='git'."
        )
elif RESOLVED_REPO_SOURCE == "git":
    ROOT = PLATFORM_REPO_PATH
    if SETUP_DRY_RUN:
        print(f"DRY RUN: would clone/update {GIT_URL}@{GIT_BRANCH} into {ROOT}")
    elif not ROOT.exists():
        print(f"Cloning {GIT_URL} branch {GIT_BRANCH} into {ROOT} ...")
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(ROOT)], check=True)
    else:
        print(f"Repo already exists at {ROOT}; checking update policy ...")
        if not is_repo_dir(ROOT):
            raise FileNotFoundError(f"Target exists but does not look like scres-ia: {ROOT}")
        if not (ROOT / ".git").exists():
            print("Existing repo copy has no .git directory; skipping git fetch/checkout. Use REPO_SOURCE='git' with an empty target for auto-update.")
        else:
            run_cmd(["git", "-C", str(ROOT), "fetch", "origin", GIT_BRANCH, "--depth", "1"], check=False)
            run_cmd(["git", "-C", str(ROOT), "checkout", GIT_BRANCH], check=False)
            if AUTO_UPDATE_REPO:
                if FORCE_RESET_REPO:
                    print("FORCE_RESET_REPO=True: resetting checkout to origin branch. Local repo edits will be lost.")
                    run_cmd(["git", "-C", str(ROOT), "reset", "--hard", f"origin/{GIT_BRANCH}"], check=False)
                else:
                    print("AUTO_UPDATE_REPO=True: attempting safe fast-forward pull.")
                    result = run_cmd(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", GIT_BRANCH], check=False)
                    if result.returncode != 0:
                        print("WARNING: safe pull failed. Set FORCE_RESET_REPO=True only in disposable Colab/Kaggle runtimes if needed.")
elif RESOLVED_REPO_SOURCE == "drive":
    ROOT = PLATFORM_REPO_PATH
    drive_path = Path(DRIVE_REPO_PATH)
    if PLATFORM == "colab" and not SETUP_DRY_RUN:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as exc:
            print("Drive mount skipped/failed:", exc)
    if SETUP_DRY_RUN:
        print(f"DRY RUN: would copy repo from Drive {drive_path} -> {ROOT}")
    else:
        if not drive_path.exists():
            raise FileNotFoundError(f"DRIVE_REPO_PATH does not exist: {drive_path}")
        if ROOT.exists():
            print("Removing existing local repo copy:", ROOT)
            shutil.rmtree(ROOT)
        print(f"Copying repo from Drive {drive_path} -> {ROOT}")
        shutil.copytree(drive_path, ROOT)
elif RESOLVED_REPO_SOURCE == "upload_zip":
    ROOT = PLATFORM_REPO_PATH
    zip_path = Path(UPLOAD_ZIP_PATH)
    if SETUP_DRY_RUN:
        print(f"DRY RUN: would unzip {zip_path} -> {ROOT.parent}")
    else:
        if not zip_path.exists():
            raise FileNotFoundError(f"UPLOAD_ZIP_PATH does not exist: {zip_path}")
        if ROOT.exists():
            print("Removing existing local repo copy:", ROOT)
            shutil.rmtree(ROOT)
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        print(f"Unzipping {zip_path} -> {ROOT.parent}")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(ROOT.parent)
        if not ROOT.exists():
            # Common case: zip contains a top-level scres-ia-main or similar directory.
            candidates = [p for p in ROOT.parent.iterdir() if p.is_dir() and is_repo_dir(p)]
            if not candidates:
                raise FileNotFoundError("Could not find extracted repo folder containing supply_chain/ and scripts/.")
            candidates[0].rename(ROOT)
else:
    raise ValueError(f"Unknown REPO_SOURCE={REPO_SOURCE!r}")

ROOT = ROOT.resolve()
if not SETUP_DRY_RUN and not is_repo_dir(ROOT):
    raise FileNotFoundError(f"Resolved ROOT does not look like scres-ia: {ROOT}")

os.chdir(ROOT)
# Insert both handles. Some legacy notebooks imported `scresia.supply_chain`, while
# the current repo imports `supply_chain`. Keeping both makes local/Colab/Kaggle robust.
for path in (ROOT.parent, ROOT):
    s = str(path)
    if s not in sys.path:
        sys.path.insert(0, s)

OUTPUT_ROOT = ROOT / OUTPUT_ROOT if not OUTPUT_ROOT.is_absolute() else OUTPUT_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT handle:", ROOT)
print("Working directory:", Path.cwd())
print("Output root:", OUTPUT_ROOT)
print("sys.path head:", sys.path[:5])
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform details:", platform.platform())
if (ROOT / ".git").exists():
    print("Branch/status:")
    run_cmd(["git", "branch", "--show-current"], check=False)
    run_cmd(["git", "rev-parse", "--short", "HEAD"], check=False)
else:
    print("Branch/status: not available because this repo copy has no .git directory.")

if UPDATE_NOTEBOOK_COPY and (ROOT / '.git').exists() and not SETUP_DRY_RUN:
    target = ROOT / NOTEBOOK_RELATIVE_PATH
    try:
        print("Refreshing notebook file from fetched git branch into", target)
        target.parent.mkdir(parents=True, exist_ok=True)
        result = subprocess.run(
            ["git", "-C", str(ROOT), "show", f"origin/{GIT_BRANCH}:{NOTEBOOK_RELATIVE_PATH}"],
            check=True,
            capture_output=True,
            text=False,
        )
        target.write_bytes(result.stdout)
        print("Notebook copy updated on disk. Reopen notebook UI to see updated cells.")
    except Exception as exc:
        print("Notebook copy update skipped/failed:", exc)

if INSTALL_DEPS and not SETUP_DRY_RUN:
    req = ROOT / "requirements.txt"
    if req.exists():
        print("Installing requirements.txt. This may take a few minutes on a fresh Colab/Kaggle runtime.")
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
    else:
        print("requirements.txt not found; installing minimal runtime packages.")
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "simpy", "gymnasium", "stable-baselines3", "sb3-contrib", "einops", "openpyxl"], check=False)


## Required-file preflight

Before training, the notebook checks that the repo is fresh enough for David's lab. If any file is missing, stop and update the repo or switch to the correct branch.

In [ ]:
# ============================================================
# Required-file preflight
# ============================================================
REQUIRED_PATHS = [
    "scripts/david_dkana_thesis_faithful_smoke.py",
    "scripts/export_trajectories_for_david.py",
    "scripts/build_dkana_dataset.py",
    "scripts/train_dkana_behavior_clone.py",
    "scripts/run_track_b_smoke.py",
    "scripts/run_track_b_adaptive_sweep.py",
    "supply_chain/external_env_interface.py",
    "supply_chain/dkana_env.py",
    "supply_chain/episode_metrics.py",
]
missing = [path for path in REQUIRED_PATHS if not (ROOT / path).exists()]
if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError("Repo preflight failed. Update the repo or switch to the correct branch before running experiments.")
print("Preflight passed. Required files are present:")
for path in REQUIRED_PATHS:
    print(" -", path)

## Decision variables explained

### Track A variables

Track A asks whether RL can improve using Garrido-style decisions.

There are several Track A contracts in this repo:

1. **Thesis faithful DKANA 18D / 19D contract**
   - Action vector has 18 entries.
   - First 15 entries score inventory-period choices:
     - Op3: indices `0..4` for inventory periods `I168, I336, I504, I672, I1344`.
     - Op5: indices `5..9`.
     - Op9: indices `10..14`.
   - Last 3 entries score shifts:
     - `S1`: index `15`.
     - `S2`: index `16`.
     - `S3`: index `17`.
   - Default observation mode `decision_reward` is 19D: realized 18D decision vector plus reward.
   - This is the best handoff contract for David if the goal is thesis-faithful DKANA/DMLPA compatibility.

2. **Continuous Track A / per-op buffer contract**
   - Action may be `[common_buffer_frac, shift_signal]`, or richer `[op3_frac, op5_frac, op9_frac, shift_signal]`.
   - `shift_signal < -0.33` maps to `S1`, between `-0.33` and `0.33` maps to `S2`, and above `0.33` maps to `S3`.
   - This is useful for testing if PPO can find a narrow static plateau.

Recommended Track A audit defaults:

- Do not start with blind PPO.
- First run a static headroom gate.
- If there is no oracle-vs-constant headroom, do not train PPO.
- If there is headroom, use behavior cloning from the best static / oracle and then PPO fine-tuning.

### Track B variables

Track B uses `track_b_v1`, an 8D continuous action in `[-1, 1]`:

| Index | Variable | Meaning |
|---:|---|---|
| 0 | Op3 Q signal | upstream inventory/dispatch multiplier |
| 1 | Op9 Q signal | downstream supply-base quantity multiplier |
| 2 | Op3 ROP signal | reorder point extension |
| 3 | Op9 ROP signal | reorder point extension |
| 4 | Op5 Q signal | assembly input quantity multiplier |
| 5 | shift signal | maps to S1/S2/S3 via thresholds |
| 6 | Op10 dispatch signal | downstream dispatch multiplier, key Track B lever |
| 7 | Op12 dispatch signal | last-mile dispatch multiplier, key Track B lever |

The important Track B levers are indices `6` and `7` because they touch downstream dispatch and recovery. That is where the confirmed win lives.

In [ ]:
# ============================================================
# 2) Imports used by both Track A and Track B cells
# ============================================================
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from einops import rearrange
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from stable_baselines3 import PPO
try:
    from stable_baselines3 import SAC
except Exception:
    SAC = None
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from supply_chain.external_env_interface import (
    get_track_b_env_spec,
    get_dkana_thesis_faithful_env_spec,
    make_track_b_env,
    make_dkana_thesis_faithful_env,
    get_episode_terminal_metrics,
    get_observation_fields,
)
from supply_chain.continuous_its_env import (
    make_per_op_buffer_track_a_env,
    make_per_op_buffer_multidiscrete_track_a_env,
)
from supply_chain.episode_metrics import compute_episode_metrics

print("Imports OK")


In [ ]:
# ============================================================
# 3) Quick tests: environment contracts and one-step execution
# ============================================================
# These are intentionally short. They prove that the repo, imports, action spaces,
# and observation spaces are wired correctly before we spend time training.


def smoke_track_a_thesis_faithful() -> dict[str, Any]:
    env = make_dkana_thesis_faithful_env(
        reward_mode="ReT_seq_v1",
        risk_level="increased",
        observation_mode="decision_reward",
        max_steps=2,
        stochastic_pt=True,
    )
    obs, info = env.reset(seed=123)
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    # A simple thesis-style action: choose one inventory period for Op3/Op5/Op9 and S2.
    # This is not claimed optimal; it is just a contract smoke test.
    action[2] = 1.0   # Op3 I504,1 score
    action[7] = 1.0   # Op5 I504,1 score
    action[12] = 1.0  # Op9 I504,1 score
    action[16] = 1.0  # S2 score
    next_obs, reward, terminated, truncated, info = env.step(action)
    result = {
        "action_shape": tuple(env.action_space.shape),
        "observation_shape": tuple(env.observation_space.shape),
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "action_contract": getattr(env, "action_contract", "dkana_thesis_faithful"),
    }
    env.close()
    return result


def smoke_track_b() -> dict[str, Any]:
    env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
    obs, info = env.reset(seed=456)
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    next_obs, reward, terminated, truncated, info = env.step(action)
    result = {
        "action_shape": tuple(env.action_space.shape),
        "observation_shape": tuple(env.observation_space.shape),
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "observation_version": TRACK_B_DEFAULTS["observation_version"],
        "reward_mode": TRACK_B_DEFAULTS["reward_mode"],
        "risk_level": TRACK_B_DEFAULTS["risk_level"],
    }
    env.close()
    return result

track_a_smoke = smoke_track_a_thesis_faithful()
track_b_smoke = smoke_track_b()
print("Track A thesis-faithful smoke:")
print(json.dumps(track_a_smoke, indent=2))
print("\nTrack B smoke:")
print(json.dumps(track_b_smoke, indent=2))

assert track_a_smoke["action_shape"] == (18,), "Track A thesis-faithful action should be 18D"
assert track_b_smoke["action_shape"] == (8,), "Track B action should be 8D"
print("Smoke tests passed")

# Part I — Track A Work Lab

Track A comes first because it is the Garrido-original decision family. Use this section to audit the thesis-faithful 18D/19D contract and to ask whether Track A has any dynamic headroom left.

Default stance today:

- Track A is **not** the headline positive lane.
- It is a boundary-characterization lane.
- Run static/oracle gates before training anything expensive.

## Track A audit cells

Track A is not the current positive lane, but David may want to test it.

Use these cells to:

1. verify the thesis-faithful DKANA 18D/19D contract;
2. run a small static headroom gate;
3. optionally run the repair PPO if a gate directory exists.

Do not spend large compute on Track A unless a static headroom gate first shows that oracle-per-regime beats best-constant. If static does not show headroom, PPO should not be expected to win.

In [ ]:
# ============================================================
# 8) Track A thesis-faithful DKANA smoke command
# ============================================================
# This is the cleanest David-facing Track A compatibility check.

if RUN_TRACK_A_DKANA_SMOKE:
    cmd = [
        sys.executable, "scripts/david_dkana_thesis_faithful_smoke.py",
        "--observation-mode", "decision_reward",
        "--reward-mode", "ReT_seq_v1",
        "--risk-level", "increased",
        "--max-steps", "2",
        "--seed", "42",
        "--stochastic-pt",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped Track A DKANA smoke.")

## Track A DKANA export lab

This section is borrowed from the thesis-faithful David notebooks. It lets David export trajectories, build DKANA temporal windows, and optionally train a behavior-cloning DKANA model.

Use it when the goal is architecture/data work, not a Track B downstream-control experiment.

In [ ]:
# ============================================================
# Track A DKANA trajectory export
# ============================================================
# Writes observations/actions/rewards and metadata for David-style offline training.

if RUN_TRACK_A_DKANA_EXPORT:
    mode_tag = "19d" if DKANA_OBSERVATION_MODE == "decision_reward" else "sdm"
    EXPORT_DIR = OUTPUT_ROOT / f"data_export_thesis_dkana_{mode_tag}"
    export_cmd = [
        sys.executable, "scripts/export_trajectories_for_david.py",
        "--episodes", str(DKANA_EPISODES),
        "--seed-start", str(DKANA_SEED_START),
        "--reward-mode", DKANA_REWARD_MODE,
        "--risk-level", DKANA_RISK_LEVEL,
        "--action-contract", "thesis_faithful_dkana_v1",
        "--thesis-observation-mode", DKANA_OBSERVATION_MODE,
        "--thesis-inventory-period-mode", "thesis_strict",
        "--max-steps", str(MAX_STEPS),
        "--output-dir", str(EXPORT_DIR),
    ]
    if DKANA_OBSERVATION_MODE != "decision_reward":
        export_cmd.extend(["--observation-version", DKANA_OBSERVATION_VERSION])
    if DKANA_STOCHASTIC_PT:
        export_cmd.append("--stochastic-pt")
    print("Running:", " ".join(export_cmd))
    subprocess.run(export_cmd, check=True)
    print("Export directory:", EXPORT_DIR)
else:
    EXPORT_DIR = OUTPUT_ROOT / "data_export_thesis_dkana_sdm"
    print("Skipped DKANA trajectory export. Set RUN_TRACK_A_DKANA_EXPORT=True to run it.")

In [ ]:
# ============================================================
# Build DKANA temporal windows and verify shapes
# ============================================================
if RUN_TRACK_A_DKANA_BUILD_WINDOWS:
    if not EXPORT_DIR.exists():
        raise FileNotFoundError(f"EXPORT_DIR does not exist: {EXPORT_DIR}. Run trajectory export first.")
    build_cmd = [
        sys.executable, "scripts/build_dkana_dataset.py",
        "--input-dir", str(EXPORT_DIR),
        "--window-size", str(DKANA_WINDOW_SIZE),
        "--relation-mode", DKANA_RELATION_MODE,
    ]
    if DKANA_INCLUDE_PREV_REWARD:
        build_cmd.append("--include-prev-reward")
    print("Running:", " ".join(build_cmd))
    subprocess.run(build_cmd, check=True)
    DATASET_DIR = EXPORT_DIR / f"dkana_seq_w{DKANA_WINDOW_SIZE}"
    arrays = {
        "dkana_row_matrices": np.load(DATASET_DIR / "dkana_row_matrices.npy"),
        "dkana_config_context": np.load(DATASET_DIR / "dkana_config_context.npy"),
        "dkana_action_targets": np.load(DATASET_DIR / "dkana_action_targets.npy"),
        "dkana_time_mask": np.load(DATASET_DIR / "dkana_time_mask.npy"),
    }
    print("Dataset directory:", DATASET_DIR)
    for name, arr in arrays.items():
        print(name, arr.shape, arr.dtype)
    meta = json.loads((DATASET_DIR / "metadata.json").read_text(encoding="utf-8"))
    print("metadata keys:", sorted(meta.keys())[:20])
else:
    DATASET_DIR = OUTPUT_ROOT / f"data_export_thesis_dkana_sdm/dkana_seq_w{DKANA_WINDOW_SIZE}"
    print("Skipped DKANA window build. Set RUN_TRACK_A_DKANA_BUILD_WINDOWS=True to run it.")

In [ ]:
# ============================================================
# Optional DKANA behavior-cloning trainer
# ============================================================
# This calls the repository trainer. Keep it off for smoke unless David explicitly wants offline BC.

if RUN_TRACK_A_DKANA_TRAIN_BC:
    if not DATASET_DIR.exists():
        raise FileNotFoundError(f"DATASET_DIR does not exist: {DATASET_DIR}. Build DKANA windows first.")
    train_cmd = [
        sys.executable, "scripts/train_dkana_behavior_clone.py",
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", str(OUTPUT_ROOT / "dkana_bc_model"),
        "--epochs", "3" if RUN_PROFILE == "smoke" else "25",
        "--batch-size", "64",
    ]
    print("Running:", " ".join(train_cmd))
    subprocess.run(train_cmd, check=True)
else:
    print("Skipped DKANA BC training. Set RUN_TRACK_A_DKANA_TRAIN_BC=True to run it.")

In [ ]:
# ============================================================
# 9) Track A static headroom gate
# ============================================================
# This is the correct first question for Track A:
# Does a per-regime oracle beat the best single static action?
# If no, do not launch blind PPO.

if RUN_TRACK_A_HEADROOM_GATE:
    gate_out = OUTPUT_ROOT / "track_a_headroom_gate"
    cmd = [
        sys.executable, "scripts/run_track_a_headroom_search.py",
        "--mode", TRACK_A_DEFAULTS["mode"],
        "--families", TRACK_A_DEFAULTS["families"],
        "--phis", TRACK_A_DEFAULTS["phis"],
        "--psis", TRACK_A_DEFAULTS["psis"],
        "--fracs", TRACK_A_DEFAULTS["fracs"],
        "--shifts", TRACK_A_DEFAULTS["shifts"],
        "--seeds", ",".join(str(s) for s in TRAIN_SEEDS),
        "--max-steps", str(MAX_STEPS),
        "--output", str(gate_out),
    ]
    if TRACK_A_DEFAULTS["quick"]:
        cmd.append("--quick")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Gate output:", gate_out)
else:
    print("Skipped Track A headroom gate. Set RUN_TRACK_A_HEADROOM_GATE=True to run it.")

In [ ]:
# ============================================================
# 10) Optional Track A repair PPO, only after a gate exists
# ============================================================
# This reproduces the most sensible Track A repair logic:
# best-static teacher + behavior cloning + PPO fine-tune + checkpoint selection.
# It is disabled by default because Track A is not the current positive lane.

TRACK_A_GATE_DIR = OUTPUT_ROOT / "track_a_headroom_gate"

if RUN_TRACK_A_REPAIR:
    if not TRACK_A_GATE_DIR.exists():
        raise FileNotFoundError(f"Gate dir does not exist: {TRACK_A_GATE_DIR}. Run the headroom gate first.")
    repair_out = OUTPUT_ROOT / "track_a_repair"
    cmd = [
        sys.executable, "scripts/run_track_a_training_repair.py",
        "--gate-dir", str(TRACK_A_GATE_DIR),
        "--reward-mode", TRACK_A_DEFAULTS["reward_mode"],
        "--cvar-alpha", str(TRACK_A_DEFAULTS["cvar_alpha"]),
        "--seeds", ",".join(str(s) for s in TRAIN_SEEDS),
        "--n-envs", str(N_ENVS),
        "--timesteps", str(TRAIN_TIMESTEPS),
        "--bc-epochs", "20" if RUN_PROFILE == "smoke" else "150",
        "--action-mode", "multidiscrete",
        "--teacher", "best_static",
        "--max-steps", str(MAX_STEPS),
        "--learning-rate", "0.0001",
        "--output", str(repair_out),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Repair output:", repair_out)
else:
    print("Skipped Track A repair. Set RUN_TRACK_A_REPAIR=True after a gate passes.")

# Part II — Track B Work Lab

Track B comes second. This is the current positive lane because it exposes downstream dispatch controls at the bottleneck.

Default stance today:

- Use `track_b_v1`, `v7`, `control_v1`, `adaptive_benchmark_v2` for canonical reproduction.
- Use `v8`/`v9` and alternative rewards for exploration only.
- The primary metric is Excel/order-level ReT; cost, CVaR, CD, and service metrics are reported separately.

## Observation versions for Track B

Track B uses observation versions `v7`, `v8`, and `v9` most often.

- `v7` is the canonical confirmed baseline. It includes downstream bottleneck signals, rolling fill/backorder, and hazard features for R22/R23/R24.
- `v8` adds realized risk-ID observability: active/recent/duration features for R11-R14, R21-R24, and R3.
- `v9` adds queue health and trend features, including backorder queue count, unattended orders, oldest backorder age, EWMA fill/backlog, deltas, and previous produced/delivered/available hours.

Use `v7` when reproducing the current confirmed claim. Use `v8` or `v9` when searching for more headroom.

In [ ]:
# Inspect observation fields. This helps David know exactly what the model sees.
for obs_version in ["v7", "v8", "v9"]:
    fields = get_observation_fields(obs_version)
    print(f"\n{obs_version}: {len(fields)} fields")
    print(fields[:12], "...")
    print(fields[-12:])

## Editable model lab: PPO MLP, David DMLPA, positional DMLPA, and SAC

The Work Lab now exposes explicit model variants:

- `ppo_mlp`: baseline PPO MLP. This should always be included when comparing architectures.
- `ppo_dmlpa_faithful`: David's original DMLPA-style architecture: `Linear -> GELU -> Linear -> TransformerEncoder -> last token`.
- `ppo_dmlpa_positional`: same family but with sinusoidal positional encoding and LayerNorm.
- `sac_mlp`: SAC baseline for continuous Track B actions.

Use DMLPA variants as **architecture ablations**, not as the paper's main claim unless they beat the same PPO baseline under the same contract and seeds.

In [ ]:
class FriendDMLPAFaithful(BaseFeaturesExtractor):
    """David's original DMLPA-style feature extractor.

    It intentionally does not add explicit positional encoding. Use this as the
    faithful architecture baseline when comparing against David's proposed model.
    """

    def __init__(self, observation_space, factor: int = 1, features_dim: int = 120, hidden_dim: int = 100, nhead: int = 12, num_layers: int = 4):
        super().__init__(observation_space, features_dim)
        flat_dim = int(observation_space.shape[0])
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dimension {flat_dim} is not divisible by factor={factor}")
        if features_dim % nhead != 0:
            raise ValueError("features_dim must be divisible by nhead")
        self.obs_dimension = flat_dim // factor
        self.factor = factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, features_dim),
        )
        layer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=nhead, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        observations = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        observations = self.latent_rw(observations)
        observations = self.accumulated(observations)
        return observations[:, -1, :]


class FriendDMLPAPositional(BaseFeaturesExtractor):
    """DMLPA variant with sinusoidal positional encoding and LayerNorm."""

    def __init__(self, observation_space, factor: int = 1, features_dim: int = 120, hidden_dim: int = 100, nhead: int = 12, num_layers: int = 4):
        super().__init__(observation_space, features_dim)
        flat_dim = int(observation_space.shape[0])
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dimension {flat_dim} is not divisible by factor={factor}")
        if features_dim % nhead != 0:
            raise ValueError("features_dim must be divisible by nhead")
        self.obs_dimension = flat_dim // factor
        self.factor = factor
        self.features_dim = features_dim
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        layer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=nhead, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.register_buffer("pos_encoding", self.build_sinusoidal_pe(factor, features_dim))

    @staticmethod
    def build_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        observations = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        observations = self.latent_rw(observations)
        observations = observations + self.pos_encoding
        observations = self.pre_norm(observations)
        observations = self.accumulated(observations)
        return observations[:, -1, :]


# Backward-compatible alias used by older cells/notebooks.
DMLPA = FriendDMLPAPositional


def build_policy_kwargs(model_kind: str) -> dict[str, Any] | None:
    if model_kind in {"ppo_mlp", "sac_mlp"}:
        return None
    if model_kind == "ppo_dmlpa_faithful":
        extractor = FriendDMLPAFaithful
    elif model_kind in {"ppo_dmlpa", "ppo_dmlpa_positional"}:
        extractor = FriendDMLPAPositional
    else:
        raise ValueError(f"Unknown MODEL_KIND={model_kind}")
    return {
        "features_extractor_class": extractor,
        "features_extractor_kwargs": {
            "factor": DMLPA_FACTOR,
            "features_dim": DMLPA_FEATURES_DIM,
            "hidden_dim": DMLPA_HIDDEN,
            "nhead": DMLPA_NHEAD,
            "num_layers": DMLPA_NUM_LAYERS,
        },
        "net_arch": dict(pi=[128, 64], vf=[128, 64]),
    }

print("Model variants ready:", MODEL_KINDS_TO_RUN)

In [ ]:
# ============================================================
# 4) Quick architecture shape test for all selected model kinds
# ============================================================
env_for_shape = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
for model_kind in MODEL_KINDS_TO_RUN:
    print("\nChecking", model_kind)
    policy_kwargs = build_policy_kwargs(model_kind)
    if policy_kwargs and "features_extractor_class" in policy_kwargs:
        extractor_cls = policy_kwargs["features_extractor_class"]
        extractor = extractor_cls(env_for_shape.observation_space, **policy_kwargs["features_extractor_kwargs"])
        sample = torch.as_tensor(env_for_shape.observation_space.sample()[None, :], dtype=torch.float32)
        out = extractor(sample)
        print("extractor:", extractor_cls.__name__, "output shape:", tuple(out.shape))
    else:
        print("standard MLP policy; no custom extractor shape test needed")
env_for_shape.close()

## Notebook-native Track B training sandbox

This is the most editable part of the notebook.

Use this when David wants to change:

- PPO vs DMLPA feature extractor;
- learning rate;
- horizon;
- reward mode;
- observation version;
- number of seeds;
- timesteps;
- network size.

This sandbox is intentionally lightweight. It is not the final paper evaluator. It tells us whether a model can train and whether rewards/metrics move in a plausible direction.

For paper-grade evidence, use `scripts/run_track_b_smoke.py` or the dense CRN audit scripts.

In [ ]:
# ============================================================
# 5) Train one or more Track B models inside the notebook
# ============================================================
import time


def make_track_b_training_env(seed: int):
    def _factory():
        env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
        env.reset(seed=seed)
        return Monitor(env)
    return _factory


def find_vecnormalize(vec_env):
    current = vec_env
    seen = set()
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        if isinstance(current, VecNormalize):
            return current
        current = getattr(current, "venv", None)
    return None


def count_trainable_parameters(model) -> int:
    return int(sum(p.numel() for p in model.policy.parameters() if p.requires_grad))


def train_one_track_b_seed(seed: int, model_kind: str):
    env_fns = [make_track_b_training_env(seed * 10_000 + i) for i in range(N_ENVS)]
    vec_env = DummyVecEnv(env_fns)
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    if model_kind.startswith("ppo_dmlpa") and DMLPA_FACTOR > 1:
        vec_env = VecFrameStack(vec_env, n_stack=DMLPA_FACTOR)

    policy_kwargs = build_policy_kwargs(model_kind)
    start = time.time()
    if model_kind == "sac_mlp":
        if SAC is None:
            raise ImportError("SAC is unavailable. Install stable-baselines3 with SAC support.")
        model = SAC(
            "MlpPolicy",
            vec_env,
            learning_rate=TRACK_B_PPO_DEFAULTS["learning_rate"],
            batch_size=min(TRACK_B_PPO_DEFAULTS["batch_size"], 64),
            gamma=TRACK_B_PPO_DEFAULTS["gamma"],
            policy_kwargs=policy_kwargs,
            verbose=1,
            seed=seed,
        )
    else:
        model = PPO(
            "MlpPolicy",
            vec_env,
            learning_rate=TRACK_B_PPO_DEFAULTS["learning_rate"],
            n_steps=min(TRACK_B_PPO_DEFAULTS["n_steps"], 64 if RUN_PROFILE == "smoke" else TRACK_B_PPO_DEFAULTS["n_steps"]),
            batch_size=min(TRACK_B_PPO_DEFAULTS["batch_size"], 64),
            n_epochs=3 if RUN_PROFILE == "smoke" else TRACK_B_PPO_DEFAULTS["n_epochs"],
            gamma=TRACK_B_PPO_DEFAULTS["gamma"],
            gae_lambda=TRACK_B_PPO_DEFAULTS["gae_lambda"],
            clip_range=TRACK_B_PPO_DEFAULTS["clip_range"],
            policy_kwargs=policy_kwargs,
            verbose=1,
            seed=seed,
        )
    model.learn(total_timesteps=TRAIN_TIMESTEPS)
    elapsed = time.time() - start
    return model, vec_env, elapsed

trained_models = []
training_rows = []
if RUN_TRACK_B_NOTEBOOK_SANDBOX:
    for model_kind in MODEL_KINDS_TO_RUN:
        for seed in TRAIN_SEEDS:
            print(f"\nTraining Track B model={model_kind}, seed={seed}, timesteps={TRAIN_TIMESTEPS}")
            model, vec_env, elapsed = train_one_track_b_seed(seed, model_kind)
            run_dir = OUTPUT_ROOT / "models" / f"{model_kind}_seed{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)
            model_path = run_dir / "model.zip"
            vecnorm_path = run_dir / "vecnormalize.pkl"
            model.save(str(model_path))
            norm = find_vecnormalize(vec_env)
            if norm is not None:
                norm.save(str(vecnorm_path))
            trained_models.append({"policy": model_kind, "seed": seed, "model": model, "vec_env": vec_env, "run_dir": run_dir})
            training_rows.append({
                "policy": model_kind,
                "train_seed": seed,
                "train_timesteps": TRAIN_TIMESTEPS,
                "elapsed_seconds": elapsed,
                "parameter_count": count_trainable_parameters(model),
                "model_path": str(model_path),
                "vecnormalize_path": str(vecnorm_path) if vecnorm_path.exists() else "",
            })
    training_df = pd.DataFrame(training_rows)
    training_csv = OUTPUT_ROOT / "training_runs.csv"
    training_df.to_csv(training_csv, index=False)
    display(training_df)
    print("Wrote", training_csv)
else:
    print("Skipped notebook-native Track B sandbox. Set RUN_TRACK_B_NOTEBOOK_SANDBOX=True to train here.")

In [ ]:
# ============================================================
# 6) Deterministic evaluation for notebook-native models
# ============================================================
# This compact evaluator reports useful SCRES metrics and writes evaluation_results.csv.
# For full paper evidence, use the official runners and dense CRN static frontier.


def evaluate_track_b_model(model, policy_label: str, seed: int, episodes: int = EVAL_EPISODES) -> list[dict[str, Any]]:
    rows = []
    for ep in range(episodes):
        eval_seed = 90_000 + seed * 1_000 + ep
        env = make_track_b_env(**TRACK_B_ENV_DEFAULTS)
        obs, info = env.reset(seed=eval_seed)
        done = False
        total_reward = 0.0
        steps = 0
        last_info = info
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            done = bool(terminated or truncated)
            steps += 1
            last_info = info
        try:
            terminal = get_episode_terminal_metrics(env)
        except Exception as exc:
            terminal = {"terminal_metric_error": str(exc)}
        try:
            rich = compute_episode_metrics(env.unwrapped.sim)
        except Exception as exc:
            rich = {"rich_metric_error": str(exc)}
        row = {
            "policy": policy_label,
            "train_seed": seed,
            "episode": ep,
            "eval_seed": eval_seed,
            "steps": steps,
            "reward_total": total_reward,
            "terminal_fill_rate": float(terminal.get("fill_rate", np.nan)),
            "terminal_backorder_rate": float(terminal.get("backorder_rate", np.nan)),
            "order_ret_excel": float(rich.get("order_level_ret_mean", np.nan)),
            "order_ret_excel_cvar05_proxy": float(rich.get("order_level_ret_p05", np.nan)),
            "flow_fill_rate": float(rich.get("flow_fill_rate", np.nan)),
            "service_loss_auc_per_order": float(rich.get("service_loss_auc_per_order", np.nan)),
            "ctj_p99": float(rich.get("ctj_p99", np.nan)),
            "rpj_p99": float(rich.get("rpj_p99", np.nan)),
            "dpj_p99": float(rich.get("dpj_p99", np.nan)),
            "cost_proxy_last_shift": float(last_info.get("shifts_active", np.nan)),
        }
        rows.append(row)
        env.close()
    return rows

all_eval_rows = []
if trained_models:
    for item in trained_models:
        rows = evaluate_track_b_model(item["model"], policy_label=item["policy"], seed=item["seed"], episodes=EVAL_EPISODES)
        all_eval_rows.extend(rows)

    eval_df = pd.DataFrame(all_eval_rows)
    display(eval_df)
    out_csv = OUTPUT_ROOT / "evaluation_results.csv"
    eval_df.to_csv(out_csv, index=False)
    print("Wrote", out_csv)

    seed_summary = eval_df.groupby(["policy", "train_seed"], as_index=False).mean(numeric_only=True)
    policy_summary = seed_summary.groupby("policy", as_index=False).agg({
        "order_ret_excel": ["mean", "std", "count"],
        "flow_fill_rate": ["mean", "std", "count"],
        "service_loss_auc_per_order": ["mean", "std", "count"],
        "ctj_p99": ["mean", "std", "count"],
        "rpj_p99": ["mean", "std", "count"],
        "dpj_p99": ["mean", "std", "count"],
        "reward_total": ["mean", "std", "count"],
    })
    policy_summary.columns = ["_".join([str(x) for x in col if x]) for col in policy_summary.columns.to_flat_index()]
    seed_summary.to_csv(OUTPUT_ROOT / "seed_summary.csv", index=False)
    policy_summary.to_csv(OUTPUT_ROOT / "policy_summary.csv", index=False)
    display(seed_summary)
    display(policy_summary)
else:
    eval_df = pd.DataFrame()
    seed_summary = pd.DataFrame()
    policy_summary = pd.DataFrame()
    print("No trained_models in memory. Run the Track B sandbox cell or load a model first.")

## Optional analysis: bootstrap CIs, paired comparisons, effect sizes, and figures

These cells are inspired by the friend DMLPA-vs-PPO notebooks. They are intentionally optional. They work best when `eval_df` contains at least two policies and multiple seeds/episodes.

In [ ]:
# ============================================================
# Bootstrap / paired comparisons / Cohen's d
# ============================================================
from math import sqrt


def bootstrap_ci(values, reps: int = 5000, alpha: float = 0.05, seed: int = 123):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    if len(arr) == 1:
        return float(arr[0]), float(arr[0])
    rng = np.random.default_rng(seed)
    samples = rng.choice(arr, size=(reps, len(arr)), replace=True).mean(axis=1)
    return float(np.quantile(samples, alpha / 2)), float(np.quantile(samples, 1 - alpha / 2))


def cohens_d(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 2 or np.std(arr, ddof=1) == 0:
        return np.nan
    return float(np.mean(arr) / np.std(arr, ddof=1))

comparison_rows = []
if RUN_STATS_ANALYSIS and not eval_df.empty:
    metrics = ["order_ret_excel", "flow_fill_rate", "service_loss_auc_per_order", "ctj_p99", "rpj_p99", "dpj_p99", "reward_total"]
    policies = list(eval_df["policy"].dropna().unique())
    reference = "ppo_mlp" if "ppo_mlp" in policies else policies[0]
    for candidate in policies:
        if candidate == reference:
            continue
        for metric in metrics:
            left = eval_df.loc[eval_df["policy"] == candidate, ["train_seed", "episode", metric]].rename(columns={metric: "candidate"})
            right = eval_df.loc[eval_df["policy"] == reference, ["train_seed", "episode", metric]].rename(columns={metric: "reference"})
            paired = left.merge(right, on=["train_seed", "episode"], how="inner")
            if paired.empty:
                continue
            delta = paired["candidate"].astype(float) - paired["reference"].astype(float)
            lo, hi = bootstrap_ci(delta)
            comparison_rows.append({
                "candidate_policy": candidate,
                "reference_policy": reference,
                "metric": metric,
                "delta_mean": float(delta.mean()),
                "ci95_low": lo,
                "ci95_high": hi,
                "cohens_d_paired_delta": cohens_d(delta),
                "n_pairs": int(len(delta)),
            })
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(OUTPUT_ROOT / "paired_comparisons.csv", index=False)
    display(comparison_df)
    print("Wrote", OUTPUT_ROOT / "paired_comparisons.csv")
else:
    comparison_df = pd.DataFrame()
    print("Skipped stats analysis. Need eval_df and RUN_STATS_ANALYSIS=True.")

In [ ]:
# ============================================================
# Figures: quick visual diagnostics
# ============================================================
if RUN_FIGURES and not eval_df.empty:
    import matplotlib.pyplot as plt
    fig_dir = OUTPUT_ROOT / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plot_metrics = ["order_ret_excel", "flow_fill_rate", "service_loss_auc_per_order", "ctj_p99"]
    for metric in plot_metrics:
        if metric not in eval_df.columns:
            continue
        policies = list(eval_df["policy"].dropna().unique())
        data = [eval_df.loc[eval_df["policy"] == policy, metric].dropna().to_numpy() for policy in policies]
        if not any(len(x) for x in data):
            continue
        plt.figure(figsize=(max(7, 2.2 * len(policies)), 4))
        try:
            plt.boxplot(data, tick_labels=policies, showmeans=True)
        except TypeError:
            # Older Matplotlib versions used `labels`; newer versions prefer `tick_labels`.
            plt.boxplot(data, labels=policies, showmeans=True)
        plt.title(metric)
        plt.ylabel(metric)
        plt.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=20, ha="right")
        out = fig_dir / f"box_{metric}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.show()
        print("Wrote", out)
    if {"order_ret_excel", "cost_proxy_last_shift"}.issubset(eval_df.columns):
        plt.figure(figsize=(6, 4))
        for policy, group in eval_df.groupby("policy"):
            plt.scatter(group["cost_proxy_last_shift"], group["order_ret_excel"], label=policy)
        plt.xlabel("cost proxy: last active shift")
        plt.ylabel("Excel/order ReT")
        plt.title("ReT vs cost proxy")
        plt.legend()
        plt.grid(alpha=0.3)
        out = fig_dir / "scatter_ret_vs_cost_proxy.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.show()
        print("Wrote", out)
else:
    print("Skipped figures. Need eval_df and RUN_FIGURES=True.")


## Repository runner path: Track B canonical smoke/audit

The notebook-native sandbox is good for editing the model. The repository runner is better when you need reproducible artifacts:

- `summary.json`
- `summary.md`
- `episode_metrics.csv`
- `seed_metrics.csv`
- `policy_summary.csv`
- optional per-order ledger

The command below uses the current Track B defaults. For a tiny run, keep `RUN_PROFILE="smoke"`. For a more serious run, change the central config to `debug` or `serious`.

Important:

- The primary metric is Excel/order-level ReT.
- Do not judge Track B only by `fill_rate`; it can be misleading.
- Always compare against static policies under the same eval seeds.

In [ ]:
# ============================================================
# 7) Launch the official Track B runner from the notebook
# ============================================================
# Set RUN_OFFICIAL_TRACK_B = True when you want to execute this cell.

if RUN_OFFICIAL_TRACK_B:
    runner_out = OUTPUT_ROOT / "official_track_b_smoke"
    cmd = [
        sys.executable, "scripts/run_track_b_smoke.py",
        "--output-dir", str(runner_out),
        "--seeds", *[str(s) for s in TRAIN_SEEDS],
        "--train-timesteps", str(TRAIN_TIMESTEPS),
        "--eval-episodes", str(EVAL_EPISODES),
        "--reward-mode", TRACK_B_DEFAULTS["reward_mode"],
        "--risk-level", TRACK_B_DEFAULTS["risk_level"],
        "--observation-version", TRACK_B_DEFAULTS["observation_version"],
        "--max-steps", str(MAX_STEPS),
        "--n-envs", str(N_ENVS),
        "--learning-rate", str(TRACK_B_PPO_DEFAULTS["learning_rate"]),
        "--n-steps", str(TRACK_B_PPO_DEFAULTS["n_steps"]),
        "--batch-size", str(TRACK_B_PPO_DEFAULTS["batch_size"]),
    ]
    if RUN_PROFILE != "smoke":
        cmd.append("--export-order-ledger")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Runner output:", runner_out)
else:
    print("Skipped official Track B runner. Set RUN_OFFICIAL_TRACK_B=True to run it.")

## Reward and observation sweep guidance

For fast iteration, do not try every possible combination. The practical sweep we have been using is:

- Rewards:
  - `control_v1`: current canonical Track B reward; simple and surprisingly strong.
  - `ReT_excel_plus_cvar`: directly tied to Excel ReT plus tail/service-loss penalty.
  - `ReT_tail_v2`: tail/recovery-focused.
  - `ReT_garrido2024_train`: Cobb-Douglas same-bar training candidate.
- Observations:
  - `v7`: confirmed baseline.
  - `v8`: adds realized risk IDs.
  - `v9`: adds backorder queue health and trends.
- Seeds:
  - smoke/debug: 1 or 2 seeds.
  - serious screen: 2 seeds.
  - confirmatory: 5+ seeds.

Promotion rule for Track B screens:

- promote only if Excel/order ReT beats the current baseline or if CVaR/tail improves strongly;
- always report cost separately;
- do not hide failures;
- do not claim retained learning unless H4 retained-vs-reset is actually positive.

In [ ]:
# ============================================================
# 11) Optional official Track B reward/observation sweep
# ============================================================
# This is heavier than the notebook-native sandbox. Use on Kaggle/Colab after smoke tests pass.

if RUN_TRACK_B_SWEEP:
    sweep_out = OUTPUT_ROOT / "track_b_reward_obs_sweep"
    cmd = [
        sys.executable, "scripts/run_track_b_adaptive_sweep.py",
        "--reward-modes", ",".join(TRACK_B_REWARD_CANDIDATES),
        "--observation-versions", ",".join(TRACK_B_OBSERVATION_CANDIDATES),
        "--cvar-alphas", "0.05,0.1,0.2",
        "--seeds", *[str(s) for s in TRAIN_SEEDS],
        "--train-timesteps", str(TRAIN_TIMESTEPS),
        "--eval-episodes", str(EVAL_EPISODES),
        "--max-steps", str(MAX_STEPS),
        "--n-envs", str(N_ENVS),
        "--learning-rate", str(TRACK_B_PPO_DEFAULTS["learning_rate"]),
        "--output-dir", str(sweep_out),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Sweep output:", sweep_out)
else:
    print("Skipped Track B sweep. Set RUN_TRACK_B_SWEEP=True to run it.")

## How we have been doing experiments

This project has been iterative and compute-aware.

Typical pattern:

1. **Smoke test**
   - 1 seed.
   - very short horizon or very few timesteps.
   - only checks that the env/action/model wiring works.

2. **Cheap screen**
   - 1-2 seeds.
   - 20k-40k timesteps.
   - 2-6 evaluation episodes.
   - used to decide whether a lane is alive.

3. **Repair/engineering screen**
   - behavior cloning / warm-start if the target is a narrow static plateau;
   - lower learning rate (`1e-4`);
   - multiple envs (`n_envs=4` when cloud resources allow);
   - checkpoint selection when possible.

4. **Confirmatory run**
   - 5+ seeds.
   - 60k-100k timesteps or more.
   - dense static frontier under common random numbers.
   - full Garrido-style metrics panel.

Track A tried many variants:

- static dense frontiers;
- continuous buffer fractions;
- per-op buffers;
- multidiscrete shift/buffer actions;
- behavior cloning from best static;
- reward variants including Excel+CVAR and tail rewards;
- risk-family and conflict campaigns.

The practical result was that Track A often reaches a plateau but does not robustly beat the best static frontier.

Track B became the positive lane because its actions reach the downstream bottleneck. The model reduces recovery tails and backlog instead of merely adding upstream buffer.

In [ ]:
# ============================================================
# 12) Save a run manifest for reproducibility
# ============================================================
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "platform": PLATFORM,
    "git_url": GIT_URL,
    "git_branch": GIT_BRANCH,
    "run_profile": RUN_PROFILE,
    "train_seeds": TRAIN_SEEDS,
    "train_timesteps": TRAIN_TIMESTEPS,
    "eval_episodes": EVAL_EPISODES,
    "max_steps": MAX_STEPS,
    "n_envs": N_ENVS,
    "model_kind": MODEL_KIND,
    "auto_update": {
        "auto_update_repo": AUTO_UPDATE_REPO,
        "force_reset_repo": FORCE_RESET_REPO,
        "update_notebook_copy": UPDATE_NOTEBOOK_COPY,
        "notebook_relative_path": NOTEBOOK_RELATIVE_PATH,
    },
    "work_lab_toggles": {
        "run_track_a_dkana_smoke": RUN_TRACK_A_DKANA_SMOKE,
        "run_track_a_headroom_gate": RUN_TRACK_A_HEADROOM_GATE,
        "run_track_a_repair": RUN_TRACK_A_REPAIR,
        "run_track_a_dkana_export": RUN_TRACK_A_DKANA_EXPORT,
        "run_track_a_dkana_build_windows": RUN_TRACK_A_DKANA_BUILD_WINDOWS,
        "run_track_a_dkana_train_bc": RUN_TRACK_A_DKANA_TRAIN_BC,
        "run_track_b_notebook_sandbox": RUN_TRACK_B_NOTEBOOK_SANDBOX,
        "run_official_track_b": RUN_OFFICIAL_TRACK_B,
        "run_track_b_sweep": RUN_TRACK_B_SWEEP,
        "run_stats_analysis": RUN_STATS_ANALYSIS,
        "run_figures": RUN_FIGURES,
        "run_artifact_packager": RUN_ARTIFACT_PACKAGER,
    },
    "profile_ladder": PROFILE_LADDER,
    "model_kinds_to_run": MODEL_KINDS_TO_RUN,
    "dkana_export_defaults": {
        "observation_mode": DKANA_OBSERVATION_MODE,
        "observation_version": DKANA_OBSERVATION_VERSION,
        "reward_mode": DKANA_REWARD_MODE,
        "risk_level": DKANA_RISK_LEVEL,
        "episodes": DKANA_EPISODES,
        "window_size": DKANA_WINDOW_SIZE,
        "relation_mode": DKANA_RELATION_MODE,
        "include_prev_reward": DKANA_INCLUDE_PREV_REWARD,
    },
    "dmlpa": {
        "factor": DMLPA_FACTOR,
        "features_dim": DMLPA_FEATURES_DIM,
        "nhead": DMLPA_NHEAD,
        "num_layers": DMLPA_NUM_LAYERS,
        "hidden": DMLPA_HIDDEN,
    },
    "track_a_defaults": TRACK_A_DEFAULTS,
    "track_a_frozen_reference": TRACK_A_FROZEN_REFERENCE,
    "track_b_defaults": TRACK_B_DEFAULTS,
    "track_b_frozen_reference": TRACK_B_FROZEN_REFERENCE,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote manifest:", manifest_path)
print(json.dumps(manifest, indent=2)[:2500])

In [ ]:
# ============================================================
# Artifact packager
# ============================================================
# Creates a portable zip with manifests, CSVs, figures, and trained model files.
if RUN_ARTIFACT_PACKAGER:
    import zipfile
    artifact_zip = OUTPUT_ROOT / "artifacts.zip"
    with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in OUTPUT_ROOT.rglob("*"):
            if path.is_file() and path.name != artifact_zip.name:
                zf.write(path, path.relative_to(OUTPUT_ROOT))
    print("Wrote", artifact_zip)
else:
    print("Skipped artifact packaging. Set RUN_ARTIFACT_PACKAGER=True to create artifacts.zip.")

# Glossary: contracts, variables, metrics, and knobs

## Tracks

- **Track A**: Garrido-original decision family. Controls inventory/buffer and shifts. Scientifically important as the boundary case where static frontiers often capture the optimum.
- **Track B**: Operational extension. Adds downstream dispatch control, especially Op10 and Op12. Current positive lane.

## Action contracts

- **`thesis_faithful_dkana_v1`**: David/DKANA Track A handoff. 18D action, usually 19D observation when `decision_reward` is used.
- **`thesis_factorized`**: compact Track A action with inventory level and shift level.
- **`per_op_buffer`**: Track A extension with separate Op3, Op5, Op9 buffer fractions and a shift signal.
- **`track_b_v1`**: Track B 8D continuous action: Track A-like upstream controls plus Op10 and Op12 downstream dispatch signals.

## Track A decision terms

- **Op3**: upstream raw-material / early supply node.
- **Op5**: assembly input / production-related node.
- **Op9**: supply-base ration buffer near downstream flow.
- **`I168` ... `I1344`**: thesis inventory periods in hours. `I168` is about one week; `I1344` is about eight weeks.
- **`S1`, `S2`, `S3`**: one, two, or three manufacturing shifts.

## Track B decision terms

- **Op10 dispatch**: downstream movement from supply base toward CSSU / later downstream flow.
- **Op12 dispatch**: last-mile / theatre-facing dispatch. This is one of the most sensitive bottleneck levers.
- **Shift signal**: continuous action dimension mapped into S1/S2/S3 by thresholds.

## Observation versions

- **`v7`**: canonical Track B observation. Includes downstream queue pressure, rolling fill/backorder, and R22/R23/R24 hazard features.
- **`v8`**: adds realized risk-ID active/recent/duration features.
- **`v9`**: adds backorder queue health and trend features. Candidate for more headroom.

## Rewards

- **`control_v1`**: simple operational penalty reward. Current canonical Track B baseline.
- **`ReT_excel_delta`**: incremental reward aligned with Garrido Excel ReT.
- **`ReT_excel_plus_cvar`**: Excel ReT delta plus tail/service-loss penalty.
- **`ReT_tail_v2`**: recovery/tail-focused reward.
- **`ReT_garrido2024_train`**: Cobb-Douglas same-bar training candidate.

## Metrics

- **Excel/order-level ReT**: primary resilience metric for this project.
- **CVaR05**: worst-tail resilience risk, often more informative than the mean.
- **CTj**: cycle time / completion time style order metric.
- **RPj**: recovery period metric.
- **DPj**: disruption period metric.
- **APj**: absorption period metric.
- **Flow fill**: quantity-weighted service / ration-level flow performance.
- **Service-loss AUC**: accumulated service loss over time.
- **Cost index**: resource/capacity-use summary. Report separately from raw ReT.
- **CD / Cobb-Douglas**: multi-factor resilience index used as a secondary same-bar evaluation.

## Model variants

- **`ppo_mlp`**: mandatory baseline; PPO with a standard MLP policy.
- **`ppo_dmlpa_faithful`**: David's original DMLPA-style Transformer extractor, no explicit positional encoding.
- **`ppo_dmlpa_positional`**: DMLPA with sinusoidal positional encoding and LayerNorm. Use as ablation.
- **`sac_mlp`**: SAC baseline for continuous Track B actions. Useful robustness check, not current main claim.
- **`MODEL_KINDS_TO_RUN`**: list of models trained in the notebook-native sandbox. Always include `ppo_mlp` when testing architecture changes.

## Data/export terms

- **Trajectory export**: writes raw observations/actions/rewards for offline model development.
- **DKANA windows**: temporal matrices built from exported trajectories for DKANA behavior cloning.
- **`dkana_row_matrices.npy`**: temporal row matrices used by DKANA.
- **`dkana_config_context.npy`**: contextual features paired with DKANA rows.
- **`dkana_action_targets.npy`**: action labels/targets for behavior cloning.
- **`dkana_time_mask.npy`**: mask identifying valid timesteps in each temporal window.
- **`artifacts.zip`**: portable bundle with models, CSVs, figures, and manifests.

## Experimental terms

- **Smoke**: tiny run only proving code wiring.
- **Screen**: small run used to decide whether a lane is alive.
- **Confirmatory**: 5+ seeds, dense static frontier, CRN, rich metrics.
- **CRN**: common random numbers; static and learned policies are evaluated under matched random seeds.
- **Dense frontier**: many static baselines, not a coarse 3- or 5-point grid.
- **Headroom**: evidence that the best action changes by regime enough for a dynamic policy to beat the best constant policy.
- **H4 retained-vs-reset**: future retained-learning test. Do not claim memory unless retained beats reset with CI.

## Interpretation checklist for David

After a run, ask these questions:

1. Did the environment smoke tests pass?
2. Did the action shape match the intended track?
   - Track A DKANA thesis faithful: 18D.
   - Track B: 8D.
3. Did training run without NaNs or action-shape errors?
4. Did the reward curve move in a plausible direction?
5. Did Excel/order ReT improve, or only the training reward?
6. Did service tail improve: CVaR, CTj/RPj/DPj p99, service-loss AUC?
7. Did the policy use the intended levers?
   - Track A: buffer/shift.
   - Track B: Op10/Op12 dispatch and shifts.
8. Did a dense static frontier beat the learned policy?
9. Is this a smoke result, a screen result, or a confirmatory result?

For paper language:

- Say Track A is a boundary characterization, not a failure of RL in general.
- Say Track B is an operational downstream-control extension.
- Say the current mechanism is adaptive recovery / backlog control.
- Do not claim anticipation or retained learning unless the lead/lag and H4 retained-vs-reset tests support it.